In [21]:
import pandas as pd
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [9]:
data = pd.read_csv('./data/preprocessed-data/preprocessed_customer_shopping_data.csv', parse_dates=['invoice_date'])

In [10]:
X = data.drop('Sales_Revenue', axis=1)
y = data['Sales_Revenue']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Kích thước X_train: {X_train.shape}")
print(f"Kích thước X_test: {X_test.shape}")
print("\nCác cột đặc trưng (Features) còn lại trong X_train:")
print(X_train.columns.tolist())

Kích thước X_train: (78789, 6)
Kích thước X_test: (19698, 6)

Các cột đặc trưng (Features) còn lại trong X_train:
['gender', 'age', 'category', 'payment_method', 'invoice_date', 'shopping_mall']


In [12]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 78789 entries, 90788 to 15795
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   gender          78789 non-null  str           
 1   age             78789 non-null  int64         
 2   category        78789 non-null  str           
 3   payment_method  78789 non-null  str           
 4   invoice_date    78789 non-null  datetime64[us]
 5   shopping_mall   78789 non-null  str           
dtypes: datetime64[us](1), int64(1), str(4)
memory usage: 4.2 MB


In [14]:
def extract_time_features(df):
    # Tạo bản sao để không cảnh báo lỗi SettingWithCopyWarning
    df_copy = df.copy()
    
    # Bóc tách Tháng và ép về kiểu chuỗi (String) để lát nữa chạy One-Hot
    df_copy['Month'] = df_copy['invoice_date'].dt.month.astype(str)
    
    # Bóc tách Ngày trong tuần
    df_copy['DayOfWeek'] = df_copy['invoice_date'].dt.day_name()
    
    # Tạo đặc trưng phái sinh: Cuối tuần (Thứ 7, CN) = 1, Ngày thường = 0
    df_copy['Is_Weekend'] = df_copy['DayOfWeek'].isin(['Saturday', 'Sunday']).astype(int)
    
    # Xóa cột Datetime gốc vì mô hình không đọc được
    df_copy = df_copy.drop(columns=['invoice_date'])
    
    return df_copy

In [15]:
X_train = extract_time_features(X_train)
X_test = extract_time_features(X_test)

In [16]:
X_train.columns.tolist()

['gender',
 'age',
 'category',
 'payment_method',
 'shopping_mall',
 'Month',
 'DayOfWeek',
 'Is_Weekend']

In [17]:
categorical_cols = ['gender', 'category', 'payment_method', 'shopping_mall', 'Month', 'DayOfWeek']

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Chỉ fit trên train
encoder.fit(X_train[categorical_cols])

# Transform cho cả Train lẫn Test
X_train_encoded = encoder.transform(X_train[categorical_cols])
X_test_encoded = encoder.transform(X_test[categorical_cols])

# Lấy tên các cột mới (Ví dụ: gender_Female, gender_Male...)
encoded_feature_names = encoder.get_feature_names_out(categorical_cols)

# Biến mảng Numpy thành DataFrame để dễ ghép nối
X_train_encoded_df = pd.DataFrame(X_train_encoded, columns=encoded_feature_names, index=X_train.index)
X_test_encoded_df = pd.DataFrame(X_test_encoded, columns=encoded_feature_names, index=X_test.index)

# Ghép các cột One-Hot mới vào dữ liệu gốc và xóa các cột chữ cũ đi
X_train = pd.concat([X_train.drop(columns=categorical_cols), X_train_encoded_df], axis=1)
X_test = pd.concat([X_test.drop(columns=categorical_cols), X_test_encoded_df], axis=1)

In [18]:
X_train.shape

(78789, 44)

In [19]:
# Dùng cho mỗi Linear Regression
scaler = StandardScaler()

scaler.fit(X_train[['age']])

X_train['age'] = scaler.transform(X_train[['age']])
X_test['age'] = scaler.transform(X_test[['age']])

In [20]:
X_train['age'].head()

90788    1.370579
43708   -1.228972
39198   -0.429110
32303    1.703855
20351    1.170614
Name: age, dtype: float64

In [22]:
os.makedirs('./data/ready_for_train', exist_ok=True)

joblib.dump(X_train, './data/ready_for_train/X_train.pkl')
joblib.dump(X_test, './data/ready_for_train/X_test.pkl')
joblib.dump(y_train, './data/ready_for_train/y_train.pkl')
joblib.dump(y_test, './data/ready_for_train/y_test.pkl')

# Dành cho Model Deployment
joblib.dump(encoder, './data/ready_for_train/onehot_encoder.pkl')
joblib.dump(scaler, './data/ready_for_train/standard_scaler.pkl')

['./data/ready_for_train/standard_scaler.pkl']